# Retail Flow Prediction

Build an interpretable model to estimate the probability that a client sells a product within the next 24 hours.

## What the model does

The model works at position-snapshot level:

**one row = one client position at one point in time = one sell probability.**

For each snapshot, the model estimates the probability that this specific position is sold within the next 24 hours.

The same position can appear on different dates and receive a different probability each time, because market conditions, product state and client-related features change over time.

The model is trained on many historical snapshots at once. Once trained, it can be applied to the positions observed at a given date, producing one probability for each position.

These probabilities can then be ranked or aggregated to estimate expected sell activity across the book.

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
positions = pd.read_csv(
    "../data/synthetic_retail_flow_dataset.csv"
)

positions["timestamp"] = pd.to_datetime(positions["timestamp"])

positions.head()

,snapshot_id,timestamp,position_id,client,product_id,product_type,underlyings,worst_underlying,notional,issue_date,...,autocall_pct,autocall_distance,return_1d,return_5d,shock_magnitude,iv_change,client_pnl_pct,historical_client_sell_rate,recent_client_activity,sold_next_24h
0,S000001,2026-01-05 09:00:00,POS0001,Client_01,P024,Barrier Reverse Convertible,UBS,UBS,100000,2025-09-21,...,NaN,NaN,-0.011313,NaN,0.011313,0.017108,-0.009854,0.090909,2,0
1,S000002,2026-01-05 09:00:00,POS0002,Client_01,P028,Reverse Convertible,MSFT,MSFT,2000000,2025-10-16,...,NaN,NaN,-0.012231,NaN,0.012231,-0.001225,0.028557,0.090909,2,0
2,S000003,2026-01-05 09:00:00,POS0003,Client_01,P004,Callable Multi BRC,UBS|NESN|NOVN,UBS,500000,2025-07-02,...,1.0,-0.263063,-0.011313,NaN,0.011313,0.017108,-0.124650,0.090909,2,0
3,S000004,2026-01-05 09:00:00,POS0004,Client_01,P011,Callable Multi BRC,SMI|NOVN|ROG,SMI,750000,2025-07-30,...,1.0,0.001263,0.004958,NaN,0.004958,-0.015980,0.011449,0.090909,2,0
4,S000005,2026-01-05 09:00:00,POS0005,Client_01,P020,Barrier Reverse Convertible,AAPL,AAPL,250000,2025-08-21,...,NaN,NaN,-0.047460,NaN,0.047460,0.030727,-0.092173,0.090909,2,0


## Model features

Use market conditions, product state and client behaviour available at each snapshot.

In [3]:
numeric_features = [
    # market features
    "return_1d",
    "return_5d",
    "shock_magnitude",
    "iv_change",

    # product features
    "barrier_distance",
    "autocall_distance",
    "days_to_observation",
    
    # client/ position features
    "client_pnl_pct",
    "holding_days",
    "historical_client_sell_rate",
    "recent_client_activity",
    "notional"
]

categorical_features = [
    "product_type"
]

target = "sold_next_24h"

## Train and test split

In [4]:
positions = positions.sort_values("timestamp").reset_index(drop=True)

split_index = int(len(positions) * 0.80)

split_date = positions.loc[split_index, "timestamp"]

train = positions[positions["timestamp"] < split_date].copy()

test = positions[positions["timestamp"] >= split_date].copy()

In [5]:
print("Split date:", split_date)
print("Train rows:", len(train))
print("Test rows:", len(test))

print(
    "Train:", train["timestamp"].min(),
    "to", train["timestamp"].max()
)

print(
    "Test:", test["timestamp"].min(),
    "to", test["timestamp"].max()
)

Split date: 2026-03-11 09:00:00
Train rows: 5361
Test rows: 1356
Train: 2026-01-05 09:00:00 to 2026-03-10 09:00:00
Test: 2026-03-11 09:00:00 to 2026-06-19 09:00:00


In [6]:
X_train = train[numeric_features + categorical_features]
y_train = train[target]

X_test = test[numeric_features + categorical_features]
y_test = test[target]

## Preprocessing

Some values are naturally missing because certain features do not apply to every product. For example, a Reverse Convertible has no autocall distance.

Logistic Regression cannot use missing values directly, so numerical missing values are replaced with the median of the corresponding feature.

Numerical features are standardised, while `product_type` is converted into numerical dummy variables using one-hot encoding.

In [7]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), #for NaN; median is less sensitive to outliers
    ("scaler", StandardScaler()) # standardize numerical variables: z= (x-mean)/std
])

# For Product type / categorical feature
categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore") ) # transforms categories dummy variables
])

preprocessor = ColumnTransformer([

    ("numeric", numeric_transformer, numeric_features),

    ("categorical", categorical_transformer, categorical_features)
])

## Logistic Regression

In [8]:
model = Pipeline([
    ("preprocessor", preprocessor),

    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train);

In [9]:
sell_probability = model.predict_proba(X_test)[:, 1] #we only want P(sell)


## Model evaluation

In [10]:
print("Train sell rate:", y_train.mean())
print("Test sell rate:", y_test.mean())
print("Average predicted probability:", sell_probability.mean())

print("ROC-AUC:", roc_auc_score(y_test, sell_probability))
print("PR-AUC:", average_precision_score(y_test, sell_probability))


Train sell rate: 0.029099048684946838
Test sell rate: 0.03982300884955752
Average predicted probability: 0.03873971910648615
ROC-AUC: 0.7257780053478978
PR-AUC: 0.1294420957947157


- Average predicted sell probability: 3.9%, almost identical to the actual test sell rate of 4.0%.
- ROC-AUC: 0.726 → a randomly chosen sell is ranked above a randomly chosen non-sell about 73% of the time.
- PR-AUC: 0.129 vs a 0.04 baseline → sell events are concentrated substantially better than by random ranking.

In [11]:
results = test[
    [
        "timestamp",
        "client",
        "product_id",
        "product_type",
        "notional",
        "sold_next_24h"
    ]
].copy()

results["sell_probability"] = sell_probability

top_10_threshold = results["sell_probability"].quantile(0.90)

top_10 = results[
    results["sell_probability"] >= top_10_threshold
]

overall_sell_rate = results["sold_next_24h"].mean()
top_10_sell_rate = top_10["sold_next_24h"].mean()

print("Overall sell rate:", overall_sell_rate)
print("Top 10% sell rate:", top_10_sell_rate)
print("Lift:", top_10_sell_rate / overall_sell_rate)

Overall sell rate: 0.03982300884955752
Top 10% sell rate: 0.14705882352941177
Lift: 3.69281045751634


The highest-ranked 10% of positions show a 14.7% realised sell rate versus 4.0% across the full test set, corresponding to a 3.7x lift.

This supports using the model as a prioritisation tool: rather than making deterministic sell/no-sell predictions, it helps identify the positions most likely to generate secondary-market flow.

In [12]:
results["sell_probability_decile"] = pd.qcut(
    results["sell_probability"],
    q=10,
    labels=False,
    duplicates="drop"
) + 1
decile_summary = (
    results
    .groupby("sell_probability_decile")
    .agg(
        positions=("sold_next_24h", "count"),
        avg_predicted_probability=("sell_probability", "mean"),
        actual_sell_rate=("sold_next_24h", "mean")
    )
    .sort_index(ascending=False)
)

decile_summary

,positions,avg_predicted_probability,actual_sell_rate
sell_probability_decile,,,
10,136,0.173192,0.147059
9,135,0.049899,0.074074
8,136,0.036895,0.014706
7,135,0.029065,0.051852
6,136,0.024536,0.014706
5,135,0.020985,0.029630
4,136,0.018399,0.036765
3,135,0.015422,0.007407
2,136,0.011850,0.007353


Positions are grouped into ten buckets based on predicted sell probability. The highest decile contains the 10% of positions with the highest estimated probability of a sell within the next 24 hours.

This group shows a realised sell rate of 14.7%, compared with about 4.0% across the full test set.

## Expected Flow

In [13]:
print("Expected sell events:", results["sell_probability"].sum())
print("Actual sell events:", results["sold_next_24h"].sum())

results["expected_sell_notional"] = (
    results["sell_probability"] * results["notional"]
)

print(
    "Expected sell notional:",
    results["expected_sell_notional"].sum()
)

Expected sell events: 52.531059108395226
Actual sell events: 54
Expected sell notional: 43177705.39969774


## Model interpretation

In [14]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

coefficients = model.named_steps["classifier"].coef_[0]


coefficient_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coefficient_df["feature"] = (
    coefficient_df["feature"]
    .str.replace("numeric__", "", regex=False)
    .str.replace("categorical__", "", regex=False)
)

coefficient_df["abs_coefficient"] = (
    coefficient_df["coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    "abs_coefficient",
    ascending=False
)

coefficient_df[
    ["feature", "coefficient"]
].head(15)

,feature,coefficient
5,autocall_distance,-0.373414
9,historical_client_sell_rate,0.318334
10,recent_client_activity,0.311749
0,return_1d,-0.270128
17,product_type_Tracker,-0.259058
2,shock_magnitude,0.238429
7,client_pnl_pct,-0.160110
14,product_type_Bonus Certificate,0.116623
15,product_type_Callable Multi BRC,0.097726
4,barrier_distance,-0.079614


Negative client P&L and daily returns increase estimated sell probability, while historical sell propensity and recent client activity have positive coefficients. Barrier and autocall distances also contribute in economically intuitive directions.

Coefficient signs are more informative here than their exact magnitudes.

## Interpretation and limitations

### Client behaviour

Client identity is not used directly. Instead, the model uses a small set of behavioural summaries such as historical sell rate, recent activity and current client P&L.

This means that clients with different observed behaviour can receive different probabilities. However, two clients with very similar current feature values will also receive similar probabilities even if their broader trading histories are very different.

A richer implementation could build rolling features from the full transaction history: recent buy/sell frequency, typical trade size, preferred product types, average holding period, days since the last trade and historical reactions to market shocks or losses. These features would always be calculated using only information available before the snapshot date.

### Aggregating position probabilities

The model predicts one probability for each position snapshot. These probabilities can be summed to estimate the expected number of sell events across the book.

In the test period, this gives approximately **52.5 expected sells versus 54 realised sells**.

This is an expected count, not the probability that exactly 52 or 53 positions will be sold. Modelling the full distribution of future sell events would also require accounting for dependencies between positions and clients.

Expected sell notional is calculated as:

**sell probability × position notional**

This is useful as a first estimate of potential flow, but it assumes that a sell event means the full position is unwound. Real clients may sell only part of their position.

### Model and data limitations

The dataset is synthetic and relatively small, so the results demonstrate the workflow rather than realistic market volumes or real client behaviour.

Logistic Regression is deliberately simple and interpretable. It captures a linear relationship in log-odds, but does not automatically capture nonlinear effects or interactions such as the combination of a large market shock and barrier proximity.

Some missing values are also structural: for example, autocall distance does not apply to every product. The current version uses median imputation; a richer implementation could explicitly encode whether a feature is applicable.

Client behaviour and market conditions can also change over time. A production model would therefore need monitoring and periodic retraining.

The train/test split follows time: the model is trained only on earlier snapshots and evaluated on later ones. This mimics the real situation where past information is available when training, while future client behaviour is still unknown.

### Potential desk use

The main practical use would be to turn a large book of client positions into a prioritised watchlist.

For every current position, the model could estimate sell probability using the latest market, product and behavioural information. The desk could then:

- rank positions by expected likelihood of secondary-market activity;
- identify clients and products that may require attention before flow arrives;
- aggregate probabilities into expected sell-event counts;
- estimate expected sell notional;
- compare expected flow under different market-stress scenarios.

This could help a trader focus attention on the parts of the book where client flow is more likely to appear, rather than monitoring every position equally.

### Scope

The current dataset is position-based, so it naturally supports predicting SELL activity from positions already held by clients.

Secondary-market BUY flow could be modelled in a similar way, but the candidate universe would need to be broader: for each client and timestamp, it should include tradable products the client could potentially buy, not only products already held. Existing holdings could also remain candidates for additional purchases.

Combining BUY and SELL models could eventually provide an estimate of expected net client flow.

In [15]:
results["expected_sell_notional"] = (
    results["sell_probability"] * results["notional"]
)

results.to_csv(
    "../data/retail_flow_predictions.csv",
    index=False
)

import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    model,
    "../models/retail_flow_model.joblib"
)

['../models/retail_flow_model.joblib']